In [7]:
import pandas as pd
import numpy as np

from bs4 import BeautifulSoup
import re

In [3]:
df = pd.read_excel("../data/myfair_participation_message_1982.xlsx").dropna(how='all')
df.head(3)

,id,message,process_state,sender_email,message_type,participation_num
0,43031.0,"<div class=""css-0""><br data-mce-bogus=""1""></di...",부스 예약 접수 완료,customer@customer.com,ADMIN,1982
1,43034.0,"<p>(고객사 담당자명 직함)님, 안녕하세요</p><p>마이페어 (마이페어 담당자명...",참가 품목 안내,myfair@myfair.co,ADMIN,1982
2,43435.0,"<p>안녕하세요, 마이페어입니다.</p><p><br data-mce-bogus=""1...",부스 포함사항 안내,myfair@myfair.co,ADMIN,1982


In [4]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 171 entries, 0 to 170
Data columns (total 6 columns):
 #   Column             Non-Null Count  Dtype  
---  ------             --------------  -----  
 0   id                 165 non-null    float64
 1   message            165 non-null    object 
 2   process_state      81 non-null     object 
 3   sender_email       163 non-null    object 
 4   message_type       165 non-null    object 
 5   participation_num  171 non-null    int64  
dtypes: float64(1), int64(1), object(4)
memory usage: 8.1+ KB


# 영문 번역 라이브러리: Googletrans

In [6]:
from googletrans import Translator
translator = Translator()
result = translator.translate('안녕하세요', src='ko', dest='en')
print(result.text)  # "Hello"

hello


In [8]:
result = translator.translate('안녕하세요 저는 홍길동입니다. 만나서 반갑습니다.', src='ko', dest='en')
print(result.text)  # "Hello"

Hello, I am Hong Gil -dong.nice to meet you


# 전처리

In [11]:
def preprocess_html_message(html_text):
    if pd.isna(html_text):
        return ""  # 또는 NaN 유지하려면: return np.nan

    # BeautifulSoup으로 HTML 파싱
    soup = BeautifulSoup(html_text, "html.parser")

    # <br> 태그는 줄바꿈으로 대체
    for br in soup.find_all("br"):
        br.replace_with("\n")

    # 텍스트만 추출
    raw_text = soup.get_text(separator=" ", strip=True)

    # 공백 정리
    raw_text = re.sub(r'\s+', ' ', raw_text)

    return raw_text

In [12]:
df['message_clean'] = df['message'].apply(preprocess_html_message)

# 결과 출력
print(df[['message', 'message_clean']].head())

                                             message  \
0  <div class="css-0"><br data-mce-bogus="1"></di...   
1  <p>(고객사 담당자명 직함)님, 안녕하세요</p><p>마이페어 (마이페어 담당자명...   
2  <p>안녕하세요, 마이페어입니다.</p><p><br data-mce-bogus="1...   
3  <p>담당자님 안녕하세요 보내주신 내용 확인했습니다<br>부재중이셔서 채팅드립니다<...   
4  <p>네 담당자님 확인했습니다!&nbsp;혹시 부스 자리 선택 또는 2면 오픈 부스...   

                                       message_clean  
0  🎉 부스예약이 접수되었습니다. 마이페어에서 참가신청을 시작해 주셔서 감사합니다. 서...  
1  (고객사 담당자명 직함)님, 안녕하세요 마이페어 (마이페어 담당자명)입니다. 비품이...  
2  안녕하세요, 마이페어입니다. 답변 기다려 주셔서 감사합니다. 부스패키지는 아래와 같...  
3  담당자님 안녕하세요 보내주신 내용 확인했습니다 부재중이셔서 채팅드립니다 부스 안에 ...  
4  네 담당자님 확인했습니다! 혹시 부스 자리 선택 또는 2면 오픈 부스 선택 가능한가...  


In [13]:
df['message_clean'][150]

'안녕하세요, (고객사 담당자명 직함)님. 마이페어 (마이페어 담당자명 호칭)입니다. 네! 지금 바로 하실 수 있게 도와드렸습니다! 해당 절차 이후, 별도로 해주실 건 없으시며 정산에 필요한 요청 드렸던 서류 전달 주시면 됩니다. 이후로는 저희가 정산기관에 모든 서류들 모아서 제출하고 정산 받겠습니다. (약 1~2개월 소요) 감사합니다.'

# message 컬럼 번역

In [17]:
from googletrans import Translator

# Translator 객체 생성
translator = Translator()

# 결측치 제외하고 번역
def translate_text(text):
    try:
        return translator.translate(text, src='ko', dest='en').text
    except Exception as e:
        print(f"Translation failed: {e}")
        return text  # 번역 실패 시 원문 반환

# message 컬럼 영어로 번역한 새 컬럼 추가
df['message_en'] = df['message_clean'].apply(lambda x: translate_text(x) if pd.notnull(x) else x)

Translation failed: the JSON object must be str, bytes or bytearray, not NoneType
Translation failed: the JSON object must be str, bytes or bytearray, not NoneType
Translation failed: the JSON object must be str, bytes or bytearray, not NoneType
Translation failed: the JSON object must be str, bytes or bytearray, not NoneType
Translation failed: the JSON object must be str, bytes or bytearray, not NoneType
Translation failed: the JSON object must be str, bytes or bytearray, not NoneType


In [18]:
# 결과 출력
print(df[['message_clean', 'message_en']].head())

                                       message_clean  \
0  🎉 부스예약이 접수되었습니다. 마이페어에서 참가신청을 시작해 주셔서 감사합니다. 서...   
1  (고객사 담당자명 직함)님, 안녕하세요 마이페어 (마이페어 담당자명)입니다. 비품이...   
2  안녕하세요, 마이페어입니다. 답변 기다려 주셔서 감사합니다. 부스패키지는 아래와 같...   
3  담당자님 안녕하세요 보내주신 내용 확인했습니다 부재중이셔서 채팅드립니다 부스 안에 ...   
4  네 담당자님 확인했습니다! 혹시 부스 자리 선택 또는 2면 오픈 부스 선택 가능한가...   

                                          message_en  
0  🎉 booth booking has been received.Thank you fo...  
1  (Customer personnel), Hello, this is My Fair (...  
2  Hello, this is My Fair.Thank you for waiting f...  
3         Hello, I checked the contents you sent me.  
4  You have checked your person in charge!Is it p...  


In [19]:
df['message_en'][150]

'Hello, (Title of the customer).My Fair is the name of My Fair.yesI helped you to do it right now!After this procedure, you have nothing to do with you and deliver the documents you needed for settlement.Since then, we will collect and submit all the documents to the settlement agency.Thank you.'